# LangChain  Memory: forma clásica vs forma actual



In [ ]:
# Instalación recomendada para el curso
# Ejecutar una vez en un entorno limpio.

%pip install -U langchain langchain-openai langchain-core langchain-classic python-dotenv


## 1. Configuración segura

El notebook original tenía una API key hardcodeada. Eso es riesgoso: puede filtrarse, quedar en GitHub o compartirse con alumnos. Usamos `.env` o variables de entorno.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "Falta OPENAI_API_KEY. Configurala como variable de entorno o en un archivo .env"
    )

# Cambiá el modelo desde el entorno sin tocar el notebook.
MODEL_NAME = os.environ.get("OPENAI_MODEL", "gpt-5.4-nano-2026-03-17")
print("Modelo:", MODEL_NAME)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
parser = StrOutputParser()


## 2. ¿Qué es “memory” en una app conversacional?

Un modelo de chat no recuerda llamadas anteriores por sí mismo. La aplicación debe reenviar historial relevante en cada turno.

En LangChain moderno, conviene pensar en tres piezas:

1. **Historial**: dónde guardo mensajes de cada conversación.
2. **Prompt**: dónde inserto esos mensajes.
3. **Runnable con historial**: wrapper que lee y actualiza la conversación automáticamente.


## 3. Forma clásica: `ConversationChain` + `ConversationBufferMemory`


In [ ]:
# Forma vieja / legacy
# Requiere: pip install langchain-classic

from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

legacy_memory = ConversationBufferMemory()
legacy_conversation = ConversationChain(
    llm=llm,
    memory=legacy_memory,
    verbose=True,
)

legacy_conversation.predict(input="Hola, mi nombre es Julian")


In [ ]:
legacy_conversation.predict(input="¿Cómo me llamo?")


In [ ]:
legacy_memory.load_memory_variables({})


## 4. Forma nueva: `RunnableWithMessageHistory`

En lugar de `ConversationChain`, componemos una chain normal con LCEL y la envolvemos con un objeto que maneja el historial por `session_id`.


In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente claro y breve. Respondé en español."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

base_chain = prompt | llm | parser

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config_agustin = {"configurable": {"session_id": "julian"}}


In [ ]:
conversation.invoke(
    {"input": "Hola, mi nombre es Julian"},
    config=config_agustin,
)


In [ ]:
conversation.invoke(
    {"input": "¿Cómo me llamo?"},
    config=config_agustin,
)


In [ ]:
# Inspeccionar el historial guardado
for message in store["agustin"].messages:
    print(type(message).__name__ + ":", message.content)


## 5. Sesiones separadas

El `session_id` permite mantener conversaciones independientes para distintos usuarios o pestañas.


In [ ]:
config_ana = {"configurable": {"session_id": "ana"}}

print(conversation.invoke({"input": "Hola, mi nombre es Julian"}, config=config_ana))
print(conversation.invoke({"input": "¿Cómo me llamo?"}, config=config_ana))
print(conversation.invoke({"input": "¿Cómo me llamo?"}, config=config_agustin))


## 6. Equivalente moderno de `ConversationBufferWindowMemory`

La memoria de ventana guarda o usa solo los últimos `k` mensajes. En la forma moderna podemos limitar los mensajes que entran al prompt con `MessagesPlaceholder(n_messages=...)`.

> `n_messages=4` significa “usar los últimos 4 mensajes” en el prompt. El historial completo puede seguir guardado en `store`.


In [ ]:
window_prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente. Usá solo el historial visible en el prompt."),
    MessagesPlaceholder(variable_name="history", n_messages=4),
    ("human", "{input}"),
])

window_chain = window_prompt | llm | parser

window_conversation = RunnableWithMessageHistory(
    window_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config_window = {"configurable": {"session_id": "ventana-demo"}}

turnos = [
    "Mi nombre es Julia.",
    "Mi color favorito es el verde.",
    "Mi lenguaje favorito es Python.",
    "Mi comida favorita es la pizza.",
    "¿Qué recordás de mí?",
]

for t in turnos:
    print("HUMAN:", t)
    print("AI:", window_conversation.invoke({"input": t}, config=config_window))
    print("---")


## 7. Equivalente moderno de memoria con recorte por tokens

En apps reales no conviene mandar todo el historial: puede ser caro o exceder la ventana de contexto. LangChain Core incluye utilities para recortar mensajes antes de enviarlos al modelo.


In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough

# Recorta el historial antes de entrar al prompt.
# token_counter=llm usa el modelo para estimar tokens cuando está soportado.
trimmer = trim_messages(
    max_tokens=500,
    strategy="last",
    token_counter=llm,
    include_system=True,
    allow_partial=False,
)

trimmed_prompt = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente que mantiene continuidad conversacional."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

trimmed_chain = (
    RunnablePassthrough.assign(history=lambda x: trimmer.invoke(x.get("history", [])))
    | trimmed_prompt
    | llm
    | parser
)

trimmed_conversation = RunnableWithMessageHistory(
    trimmed_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

config_trimmed = {"configurable": {"session_id": "trimmed-demo"}}

trimmed_conversation.invoke({"input": "Me llamo Carla y estoy preparando un curso de IA."}, config=config_trimmed)


In [ ]:
trimmed_conversation.invoke({"input": "¿Qué estoy preparando?"}, config=config_trimmed)


## 8. Equivalente moderno de memoria con resumen

La memoria de resumen clásica resumía la conversación para ahorrar tokens. En la forma moderna conviene hacerlo explícito: mantener un resumen por sesión y actualizarlo cada ciertos turnos.


In [ ]:
summary_store = {}
message_store = {}

summary_prompt = ChatPromptTemplate.from_template(
    """
Actualizá el resumen de la conversación.

Resumen anterior:
{summary}

Nuevos mensajes:
{new_messages}

Nuevo resumen breve y útil:
"""
)
summary_chain = summary_prompt | llm | parser

chat_prompt_with_summary = ChatPromptTemplate.from_messages([
    ("system", "Sos un asistente. Resumen disponible de la conversación: {summary}"),
    MessagesPlaceholder(variable_name="history", n_messages=6),
    ("human", "{input}"),
])
chat_chain_with_summary = chat_prompt_with_summary | llm | parser

def get_summary_session_history(session_id: str):
    if session_id not in message_store:
        message_store[session_id] = InMemoryChatMessageHistory()
    return message_store[session_id]

def invoke_with_summary(user_input: str, session_id: str = "summary-demo"):
    history = get_summary_session_history(session_id)
    summary = summary_store.get(session_id, "")
    
    response = chat_chain_with_summary.invoke({
        "input": user_input,
        "history": history.messages,
        "summary": summary,
    })
    
    history.add_user_message(user_input)
    history.add_ai_message(response)
    
    # Cada 6 mensajes, actualizamos resumen y opcionalmente podríamos compactar historial.
    if len(history.messages) % 6 == 0:
        recent = "\n".join(f"{type(m).__name__}: {m.content}" for m in history.messages[-6:])
        summary_store[session_id] = summary_chain.invoke({
            "summary": summary,
            "new_messages": recent,
        })
    
    return response

print(invoke_with_summary("Me llamo Diego y estoy armando una startup de educación."))
print(invoke_with_summary("Mi objetivo es enseñar IA generativa a docentes."))
print(invoke_with_summary("¿Qué sabés de mi proyecto?"))

In [ ]:
summary_store


## 9. Comparación: memory clásica vs forma moderna

| Caso | Forma clásica | Forma moderna recomendada |
|---|---|---|
| Conversación básica | `ConversationChain` + `ConversationBufferMemory` | LCEL + `RunnableWithMessageHistory` |
| Ventana de últimos mensajes | `ConversationBufferWindowMemory(k=...)` | `MessagesPlaceholder(n_messages=...)` |
| Recorte por tokens | `ConversationTokenBufferMemory` | `trim_messages(...)` antes del prompt |
| Resumen conversacional | `ConversationSummaryMemory` | resumen explícito por sesión + prompt con `{summary}` |
| Varias sesiones | dependía de cómo se instanciaba memory | `session_id` en `config` |
| Código nuevo | no recomendado | recomendado |
